# Última Ventana — generación y EDA sintética

[Abrir este notebook en Google Colab](https://colab.research.google.com/github/v1ckybl/hackaton-equipo3/blob/feature/ultima-ventana/notebooks/01_generacion_eda_sintetica.ipynb)

Genera un dataset reproducible con el contrato v1 y revisa rangos, nulos,
clases, distribuciones y correlaciones. No requiere ningún dato externo.


In [ ]:
# COLAB_CONFIG — editar solo esta celda
REPO_URL = "https://github.com/v1ckybl/hackaton-equipo3.git"
REPO_REF = "feature/ultima-ventana"
REPO_DIR = "/content/hackaton-equipo3"
OUTPUT_ROOT = "/content/ultima_ventana_outputs/01_generacion_eda"
ROWS = 10000
RANDOM_SEED = 42
N_ESTIMATORS = 200
MIN_ROC_AUC = 0.75
DOWNLOAD_OUTPUTS = False


## 1. Preparar el runtime


In [ ]:
# COLAB_SETUP — una sola preparación idempotente por runtime
import importlib
import os
import subprocess
import sys
from pathlib import Path

repo_dir = Path(REPO_DIR)
if repo_dir.exists() and not (repo_dir / ".git").is_dir():
    raise RuntimeError(f"{repo_dir} existe pero no es un clone Git válido")
if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(repo_dir)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", str(repo_dir)], check=True)
os.chdir(repo_dir)
for module_name in list(sys.modules):
    if module_name == "ultima_ventana_ml" or module_name.startswith("ultima_ventana_ml."):
        del sys.modules[module_name]
importlib.invalidate_caches()

commit = subprocess.run(
    ["git", "-C", str(repo_dir), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
print(f"Entorno listo · Python {sys.version.split()[0]} · commit {commit}")


## 2. Generar y validar


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

from ultima_ventana_ml import (
    FEATURE_COLUMNS_V1, TARGET_NAME, create_splits,
    export_synthetic_dataset, generate_synthetic_dataset, validate_dataset,
)

output_dir = Path(OUTPUT_ROOT)
dataset_path = output_dir / "training_dataset_synthetic_v1.csv"
manifest_path = output_dir / "training_dataset_synthetic_v1_manifest.json"
dataset = generate_synthetic_dataset(ROWS, RANDOM_SEED)
validate_dataset(dataset)
splits = create_splits(dataset.target, RANDOM_SEED)
manifest = export_synthetic_dataset(dataset, splits, dataset_path, manifest_path)
display(pd.Series(manifest, name="manifest"))


## 3. Explorar calidad y distribución


In [ ]:
data = pd.read_csv(dataset_path)
assert data[list(FEATURE_COLUMNS_V1)].notna().all().all()
assert set(data[TARGET_NAME].unique()) == {0, 1}
print(f"Filas: {len(data):,} · duplicados: {data.duplicated().sum()}")
display(data.head())
display(data[list(FEATURE_COLUMNS_V1)].describe().T)
display(data.groupby("split")[TARGET_NAME].agg(filas="count", prevalencia="mean"))


In [ ]:
axes = data[list(FEATURE_COLUMNS_V1) + [TARGET_NAME]].hist(figsize=(14, 10), bins=30)
plt.suptitle("Distribuciones del dataset sintético v1")
plt.tight_layout(); plt.show()

correlation = data[list(FEATURE_COLUMNS_V1) + [TARGET_NAME]].corr()
fig, ax = plt.subplots(figsize=(9, 7))
image = ax.imshow(correlation, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(correlation)), correlation.columns, rotation=75, ha="right")
ax.set_yticks(range(len(correlation)), correlation.index)
fig.colorbar(image, ax=ax); plt.tight_layout(); plt.show()


## 4. Empaquetar dataset y manifest


In [ ]:
import shutil
archive_path = shutil.make_archive(str(output_dir), "zip", root_dir=output_dir)
print(f"Artefactos empaquetados: {archive_path}")
if DOWNLOAD_OUTPUTS:
    from google.colab import files
    files.download(archive_path)


El target y todas las features son sintéticos. Este dataset sirve para validar el software, no para medir transitabilidad real.
